# MNIST with convolutional networks

A LeNet-style CNN for handwritten digit classification. This notebook
focuses on the type-safe spatial dimension chain, where idris-ml's CNN
differs most from PyTorch's.

```
Conv2d(1->16, k=5) -> ReLU -> MaxPool(2) ->
Conv2d(16->32, k=5) -> ReLU -> MaxPool(2) ->
Dropout(0.5) -> Linear(512->10) (raw logits; loss applies log_softmax)
```

**CLI equivalent:** `make example-mnist` (requires `make download-mnist` first)


## Type-safe spatial dimensions

In PyTorch, spatial dimension mismatches are runtime errors:
```python
# PyTorch: crashes at runtime if dimensions don't match
self.fc = nn.Linear(wrong_dim, 10)  # RuntimeError!
```

In idris-ml, `ConvOutDim` and `PoolOutDim` compute output dimensions
at the type level. A dimension mismatch is a **compile error**.


In [1]:
:t conv2d

Nn.Conv.conv2d : KnownGrad g => Backend ex dt => Init (Conv2D inC outC h w kH kW padH padW (inC * (h * w)) (outC * (ConvOutDim h kH padH * ConvOutDim w kW padW)) ex dt g)


In [2]:
:t maxPool2d

Nn.Pool.maxPool2d : MaxPool2D c inH inW poolH poolW strH strW (c * (inH * inW)) (c * (PoolOutDim inH poolH strH * PoolOutDim inW poolW strW)) ex dt g


In [3]:
:t ConvOutDim


Nn.Conv.ConvOutDim : Nat -> Nat -> Nat -> Nat


In [4]:
:t PoolOutDim


Nn.Pool.PoolOutDim : Nat -> Nat -> Nat -> Nat


## Dimension chain for MNIST

Starting from 28x28 images (1 channel):

| Layer | Output H | Output W | Channels | Flat dim |
|-------|----------|----------|----------|----------|
| Input | 28 | 28 | 1 | 784 |
| Conv2d(k=5) | 24 | 24 | 16 | 9216 |
| MaxPool(2) | 12 | 12 | 16 | 2304 |
| Conv2d(k=5) | 8 | 8 | 32 | 2048 |
| MaxPool(2) | 4 | 4 | 32 | 512 |
| Linear | - | - | - | 10 |

Each dimension is computed by the type system:
- `ConvOutDim 28 5 0 = 24` (28 - 5 + 1)
- `PoolOutDim 24 2 2 = 12` ((24 - 2) / 2 + 1)

If you change the kernel size or padding, the entire chain recomputes
at compile time, and any inconsistency is caught.


## Model construction

The compiled example chains the layers into a `Seq` with `(~~>)`; the
conv/pool dims thread through the chain's types (`Example/Mnist.idr`):

```idris
mkModel : Init Model
mkModel = do
  c1 <- conv2d {inC = InC}   {outC = OutC1} {h = ImgH}     {w = ImgH}     {kH = KH} {kW = KH} {padH = 0} {padW = 0}
  c2 <- conv2d {inC = OutC1} {outC = OutC2} {h = Pool1Out} {w = Pool1Out} {kH = KH} {kW = KH} {padH = 0} {padW = 0}
  l  <- linear {i = AfterPool2} {o = NumClasses}
  pure (c1 ~~> reluA
           ~~> maxPool2d {c = OutC1} {inH = Conv1Out} {inW = Conv1Out} {poolH = 2} {poolW = 2} {strH = 2} {strW = 2}
           ~~> c2 ~~> reluA
           ~~> maxPool2d {c = OutC2} {inH = Conv2Out} {inW = Conv2Out} {poolH = 2} {poolW = 2} {strH = 2} {strW = 2}
           ~~> dropout 0.5
           ~~> l ~~> Nil)
```

The next cell builds the first conv layer live:


In [5]:
:exec run (do {
  m <- runInitL (conv2d {inC=1} {outC=16} {h=28} {w=28} {kH=5} {kW=5} {padH=0} {padW=0}
                  {ex=TapeExecutor} {dt=F64} {g=WithGrad});
  discard m; liftIO1 (putStrLn "conv2d built (1x28x28 -> 16x24x24).") })

conv2d built (1x28x28 -> 16x24x24).


## Training on real MNIST

This notebook demonstrates the architecture only. Real training uses the
MNIST dataset:

```bash
make download-mnist    # Downloads to data/mnist/
make example-mnist     # Trains LeNet (5 epochs), evaluates on test images
```

The pipeline is the standard one from the tutorials: `idxDataset` lifts the
`.idx` binary files into a `Fin`-indexed `Dataset` via the C reader,
`stream (Shuffle seed)` orders it, `batched` collates mini-batches C-side,
and `fitSupervised` trains with Adam under `NormClip 1.0`. After training,
the example converts the model with `eval` and prints argmax accuracy on a
test batch.


## Also available: 1D convolutions

idris-ml also provides `conv1d` and `maxPool1d` builders for sequence
processing. The SeqClassify example uses them to classify waveforms
(see [seq_classify.ipynb](seq_classify.ipynb)).


In [6]:
:t conv1d

Nn.Conv.conv1d : KnownGrad g => Backend ex dt => Init (Conv1D inC outC len kL pad (inC * len) (outC * ConvOutDim len kL pad) ex dt g)


## PyTorch comparison

```python
class LeNet(nn.Module):
    def __init__(self):
        self.conv1 = nn.Conv2d(1, 16, 5)
        self.conv2 = nn.Conv2d(16, 32, 5)
        self.fc = nn.Linear(512, 10)  # 512 = 32 * 4 * 4

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(-1, 512)  # <-- manual flatten, runtime error if wrong
        return self.fc(x)
```

The `x.view(-1, 512)` line is where PyTorch users commonly get runtime errors.
In idris-ml, `512 = OutC2 * (Pool2OutH * Pool2OutW)` is computed at the
type level and verified at compile time.

See `pytorch/torch_ref/scripts/mnist.py` for the full reference.


Next: [REINFORCE](reinforce.ipynb) — policy gradient reinforcement learning.
